<b><font size="6" color="#E8800A">Week 11c: Combining models for classification</font></b><br>
<b><font size="4">Training-only selection, one sealed test partition</font></b><br>

Voting fixes the combining rule. Stacking and blending fit that rule on predictions from rows the members did not train on.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>The model pool</font>](#pool)
* [<font color='#E8800A'>Voting</font>](#vote)
* [<font color='#E8800A'>Voting settings</font>](#vote_settings)
* [<font color='#E8800A'>Stacking</font>](#stack)
* [<font color='#E8800A'>Stacking settings</font>](#stack_settings)
* [<font color='#E8800A'>Blending</font>](#blend)
* [<font color='#E8800A'>Blending settings</font>](#blend_settings)
* [<font color='#E8800A'>Selection of the combiners</font>](#sel2)
* [<font color='#E8800A'>The combiner winner</font>](#combiner_winner)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

Run the boosting notebook for this task first. Its saved selection must match these data, recipe, split and folds; a missing or stale file stops execution before a combiner is fitted.

In [ ]:
# The notebook folder holds preprocessing.py and course_helpers.py, so it joins the import path first.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Libraries for data handling, figures and parallel work.
import os
import json
import time
import warnings
import joblib
from itertools import combinations
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn: the models, the validation tools and the metrics used below.
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, cross_validate, train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import FunctionTransformer
# Progress bars, then the course helpers: the figure palette, the model log and the scoreboard.
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context
# The fold-owned preparation, stacking and blending classes written for this course.
from preprocessing import FoldStacking, FoldBlending, LockedMemory, fold_owned_members, unprepared_boosting
from matplotlib_inline.backend_inline import set_matplotlib_formats

# One seed for every random choice, then the figure style: white grid, 150 dpi, vector output.
random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
set_matplotlib_formats("svg")
# The exact-message filter also reaches the parallel workers through the environment; the library message is expected, because a validation fold can hold a category the training rows lack.
os.environ["PYTHONWARNINGS"] = "ignore:Found unknown categories:UserWarning"
warnings.filterwarnings("ignore", message="Found unknown categories.*during transform", category=UserWarning)
print("scikit-learn", sklearn.__version__)
from preprocessing import load_boosting_selection

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


__Step 1:__ Load the data and prepare the same training folds.

In [ ]:
# The champions table with the feature recipe of the feature-work notebook, indexed by athlete.
recipe_log = "../../logs/week_04_feature_work_classification_log.json"
frame, recipe = load_recipe("../../data/interim/champions.csv", recipe_log)
frame = frame.set_index("RecordID")

# Features and target, then the test rows are sealed once: 20 percent of the athletes, stratified by outcome, and no choice below reads them.
features = frame.loc[:, recipe["categorical"] + recipe["numeric"]].copy()
target = frame["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(features, target,
    test_size=0.2, random_state=random_state, stratify=target)
# `prepared`, `raw_prepared` and `ada_prepared` hold, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`): prepared with the recipe, left raw, and raw but made finite for the raw-row path of calibrated naive Bayes inside AdaBoost.
preparation = FoldPreprocessor(recipe)
folds = outer_splitter("classification", random_state=random_state)
prepared = prepared_folds(preparation, list(folds.split(X_train, y_train)), X_train, y_train)
raw_prepared = prepared_folds(FunctionTransformer(), list(folds.split(X_train, y_train)), X_train, y_train)
# The recipe is fitted once on all the training rows. Every model fitted on all the rows learns from `Z_train` and is scored on `Z_test`, the same prepared rows, so no model refits the recipe.
full_prep = clone(preparation).fit(X_train, y_train)
Z_train = full_prep.transform(X_train)
Z_test = full_prep.transform(X_test)
# `model_log` collects every logged result in order; `results` is the folder of saved results and of this notebook's fitted winner.
model_log = []
results_dir = "../../results"
print("training folds:", folds.get_n_splits())

__Step 2:__ Write the earlier learners once and create the shared member cache.

In [ ]:
taught_learners = {
    "logistic regression": LogisticRegression(solver="saga", l1_ratio=1.0, C=100, fit_intercept=False, max_iter=5000, random_state=random_state),
    "naive Bayes": CalibratedClassifierCV(GaussianNB(var_smoothing=0.1, priors=None), method="isotonic"),
    "KNN": KNeighborsClassifier(n_neighbors=41, p=1, weights="distance"),
    "pruned tree": DecisionTreeClassifier(ccp_alpha=0.001, class_weight=None, criterion="gini", max_depth=None, max_features=None, min_samples_leaf=20, min_samples_split=50, splitter="best", random_state=random_state),
    "bagged trees": BaggingClassifier(estimator=DecisionTreeClassifier(random_state=33), n_estimators=139, n_jobs=4, random_state=33),
    "random forest": RandomForestClassifier(n_estimators=127, n_jobs=4, random_state=33),
}

<div class="alert alert-block alert-info">
<code>LockedMemory(location=None, backend="local", mmap_mode=None, compress=False, verbose=1, backend_options=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
A joblib `Memory` with a lock file keyed by the function and its arguments. A second worker waits for the first computation, then reads its cache entry.

**Main Parameters:**
- `location`: cache folder; `None` switches caching off.
- Other parameters: those of joblib `Memory`.

**Main Methods:**
- `cache(func)`: the cached, locked version of `func`.

**Differences from scikit-learn:**
scikit-learn uses joblib [`Memory`](https://joblib.readthedocs.io/en/latest/user_guide/memory.html), for example in `Pipeline(memory=...)`, which allows concurrent workers to compute the same missing entry. `LockedMemory` prevents repeated preprocessing fits when the `filelock` package is installed. Without it, locking is skipped and duplicate computations give the same result.
</div>

__Step 3:__ Compare the earlier model labels with their explicit constructors.

In [ ]:
# Compare the explicit constructors with the stored labels of the earlier models.
for (name, learner), week in zip(list(taught_learners.items())[:4], range(6, 10)):
    stored = json.loads(Path(results_dir, f"week_{week:02d}.json").read_text(encoding="utf-8"))["classification"]["model"]
    print(f"{name} | week_{week:02d}: {stored}")
    print("constructor:", learner)
for name, learner in list(taught_learners.items())[4:]:
    print(f"{name} | best evaluated in the previous week")
    print("constructor:", learner)
member_memory = LockedMemory(".ensemble_member_cache", verbose=0)

<div class="alert alert-block alert-info">
<code>load_boosting_selection(path, *, task, context, fold_fingerprint, n_splits)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:** Loads the saved boosting winner only when its evidence matches this notebook.

**Main Parameters:** `path` names the selection; `task`, `context`, `fold_fingerprint` and `n_splits` describe the current data and folds.

**Returns:** Selected settings, fold scores and the fitted winner.

**Differences from scikit-learn:** [joblib.load](https://joblib.readthedocs.io/en/latest/generated/joblib.load.html) restores a file. This course helper also checks the task, data, recipe, folds, array shapes and fitted-model checksum. A mismatch asks you to rerun boosting; it never starts another boosting search.
</div>

__Step 4:__ Validate and load the boosting selection before fitting a combiner.

In [ ]:
# Fingerprint the data, recipe, split and exact outer rows.
selection_context = result_context(
    "../../data/interim/champions.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train))
fold_fingerprint = joblib.hash(outer_folds(folds, X_train, y_train))
selection = load_boosting_selection(Path(results_dir) / "boosting" / "week_11_classification_selection.joblib",
    task="classification", context=selection_context, fold_fingerprint=fold_fingerprint, n_splits=folds.get_n_splits())
# Restore settings selected independently on each outer training fold.
boost_settings = selection["boost_settings"]
boost_fold_settings = selection["boost_fold_settings"]
boost_nested_scores = selection["boost_nested_scores"]
boost_nested_validation = np.asarray(selection["boost_nested_validation"])
boost_nested_train = np.asarray(selection["boost_nested_train"])
boost_test = selection["boost_test"]
boost_label = selection["boost_label"]
boosting_winner = selection["boosting_winner"]
boost_baseline_cv = selection["baseline_cv"]
boost_baseline_test = selection["baseline_test"]
# Rebuild the unfitted member; its fitted all-training winner remains separate.
boosting_member = clone(boost_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in boost_settings.items() if key.startswith("estimator__")})
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
print("Loaded boosting winner:", boost_label)
print("Outer-training selections:", len(boost_fold_settings))
# F1 selects; the same outer fits also report the two companion metrics.
companions = {"f1": "f1", "balanced_accuracy": "balanced_accuracy",
    "roc_auc": lambda search, X, y: roc_auc_score(y, search.predict_proba(X)[:, 1]) if hasattr(search.best_estimator_.estimator_, "predict_proba") else np.nan}

# <font color='#E8800A'>The model pool</font> <a class="anchor" id="pool"></a>
[Back to TOC](#toc)

Voting, stacking and blending draw their members from seven models: the six earlier models and the boosting winner. Every combiner tries every combination of two to seven of them.

<div class="alert alert-block alert-info">
<code>fold_owned_members(preprocessor, estimators, memory=None)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:**
Turns `(name, learner)` pairs into `CachedPreparedMember` objects, a `PreparedEstimator` subclass that prepares its own raw rows. With memory and fixed learner randomness, `fit` is lazy: predictions for matching fit and score rows are computed once, then cached.

**Main Parameters:**
- `preprocessor`: unfitted recipe, cloned and fitted on each member's training rows.
- `estimators`: list of `(name, learner)` pairs.
- `memory`: optional joblib `Memory` for predictions.

**Returns:**
A list of `(name, member)` pairs. `CalibratedClassifierCV` is separated into its base learner and calibration method so calibration owns the raw split. Boosted learners whose bases already prepare raw rows receive the raw frame; other learners receive cloned recipes.

**Differences from scikit-learn:**
[`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) and [`StackingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html) fit supplied learners on supplied rows. The course members own their preparation and cache answers by fit and score rows, allowing combinations to reuse predictions instead of refitting.
</div>

__Step 5:__ Write the pool of seven models, wrap each so that it prepares its own raw rows, and name the kind of each.

In [ ]:
# The seven models the combiners draw from: the six earlier models and the boosting winner.
members = list(taught_learners.items()) + [("boosting winner", boosting_member)]
# The same members, each wrapped so that it prepares its own raw rows; `member_memory` stores their predictions for reuse.
raw_members = fold_owned_members(preparation, members, memory=member_memory)
# The kind of each model, used below to count how many kinds a combination mixes.
kinds = {'logistic regression': 'linear', 'naive Bayes': 'naive Bayes', 'KNN': 'nearest neighbours', 'pruned tree': 'single tree', 'bagged trees': 'tree ensemble', 'random forest': 'tree ensemble', 'boosting winner': 'tree ensemble'}
print("pool:", [name for name, _ in members])
print("kinds:", sorted(set(kinds.values())))

**The pool holds seven models, the six earlier ones and the boosting winner, which is `HistGradientBoostingClassifier(early_stopping=False, max_bins=63, max_iter=90, max_leaf_nodes=15, min_samples_leaf=5, random_state=33)`.** The tables and figures below call it `boosting winner`; the log and the board name it by its class and these settings. The seven fall into five kinds: linear, naive Bayes, nearest neighbours, a single tree and tree ensembles.

# <font color='#E8800A'>Voting</font> <a class="anchor" id="vote"></a>
[Back to TOC](#toc)

A fixed average uses the same combination rule for every row. Classification can average probabilities or count labels. Members retain the settings of the earlier weeks. [Voting guide](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier)

__Step 6:__ Combine three toy member answers by hand before fitting a vote.

In [ ]:
toy_member_answers = np.array([0.2, 0.6, 0.7])
print("member positive probabilities:", toy_member_answers)
print("soft mean:", toy_member_answers.mean(), "hard majority:", np.mean(toy_member_answers >= 0.5) >= 0.5)
print("soft class:", int(toy_member_answers.mean() > 0.5))

**Two of the three members give a probability above 0.5, so hard voting predicts the positive class; the soft mean is exactly 0.5 and the printed soft class is 0, because a tie goes to the first class.**

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html'>sklearn.ensemble.VotingClassifier(estimators, voting='hard', weights=None, n_jobs=None)</a>

**Definition:**
Fits every member independently on the same rows and combines their answers by a fixed rule: a label count or an average of probabilities.

**Main Parameters:**
- `estimators`: the named members.
- `voting`: `hard` counts labels, `soft` averages probabilities.
- `weights`: multiplier on each member's answer.
- `n_jobs`: workers for the member fits.

**Main Attributes (after fitting):**
- `estimators_`: the fitted members.
- `named_estimators_`: the fitted members by name.

**Main Methods:**
- `fit(X, y)`: fit each member.
- `predict(X)`: the combined label.
- `predict_proba(X)`: the averaged probabilities, for `soft` voting.
</div>

The combination rule is fixed by the class: no test column chooses membership or weights.

__Step 7:__ Fit scikit-learn `VotingClassifier` with fresh members and the vote of cached members on the same rows and compare their answers.

In [ ]:
# A small member set and the first fold of raw rows, so the two votes can be compared quickly.
equal_members = [(name, taught_learners[name]) for name in ['logistic regression', 'KNN', 'pruned tree']]
equal_fold = raw_prepared[0]
# scikit-learn's vote: every member is wrapped with the recipe so it prepares its own rows.
started = time.perf_counter()
library_vote = VotingClassifier([(name, PreparedEstimator(FoldPreprocessor(recipe), clone(member))) for name, member in equal_members], voting="soft", n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
library_answer = library_vote.predict_proba(equal_fold["X_score"])
library_vote_seconds = time.perf_counter() - started
# The vote of cached members: the first fit fills the cache, the second, timed, fit reads the member predictions from it.
vote_memory = joblib.Memory(".equal_vote_cache", verbose=0)
VotingClassifier(fold_owned_members(preparation, equal_members, memory=vote_memory), voting="soft", n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"]).predict_proba(equal_fold["X_score"])
started = time.perf_counter()
cached_vote = VotingClassifier(fold_owned_members(preparation, equal_members, memory=vote_memory), voting="soft", n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
cached_answer = cached_vote.predict_proba(equal_fold["X_score"])
cached_vote_seconds = time.perf_counter() - started
# The two votes must give the same answers; stop here if they differ, and print the largest difference with the two timings.
assert np.array_equal(library_answer, cached_answer), np.abs(library_answer - cached_answer).max()
print("held-out answers are identical:", np.array_equal(library_answer, cached_answer), "| largest difference", np.abs(library_answer - cached_answer).max())
print(f"VotingClassifier {library_vote_seconds:.2f} s, the vote of cached members {cached_vote_seconds:.2f} s")

**The two votes reproduce the same held-out predictions within the stated tolerance.** Caching reuses each member's answers without changing the vote.

__Step 8:__ Declare the member combinations and the weight alternatives before seeing any combination scores.

The seven models give 120 combinations of two to seven members, excluding single-model combiners. Each member's fold predictions are reused across votes, stacks and blends. Combination sweeps use the first three of ten training folds; both selections use all ten. Weights vary only for the full pool. Rather than refitting [`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) for each combination, the vote combines cached predictions. The equality check above compares this result with fresh-member fits.

In [ ]:
combiner_folds = raw_prepared[:3]
member_index = [combo for size in range(2, len(members) + 1) for combo in combinations(range(len(members)), size)]
member_values = [[members[i] for i in combo] for combo in member_index]
weight_values = [None] + [tuple(2 if j == i else 1 for j in range(len(members))) for i in range(len(members))]
vote_reference = VotingClassifier(estimators=raw_members, voting="soft", n_jobs=4)

__Step 9:__ Score the VotingClassifier reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(vote_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
vote_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
vote_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 10:__ Fit the VotingClassifier reference on all the training rows.

In [ ]:
vote_default = clone(vote_reference).fit(X_train, y_train)
vote_default_configuration = f"VotingClassifier(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in vote_reference.estimators)}], voting={vote_reference.voting!r}" + ("" if vote_reference.weights is None else f", weights={vote_reference.weights}") + ")"

__Step 11:__ Log the VotingClassifier reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_default_configuration,
    "train": vote_default_train, "validation": vote_default_validation, "model": vote_default},
    X_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Voting settings</font> <a class="anchor" id="vote_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the vote and scores every value on the first three training folds, because every combination refits all of its members on each fold. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members whose answers are combined. |
| `weights` | Multiplier on each member's answer. |
| `voting` | Hard label count or soft probability average (classification only). |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)

__Step 12:__ Change only voting `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `voting: estimators`

Voting fits each member independently and combines their predictions. There is no default member list: the full pool below is the fixed reference. Different errors may make a member useful, but adding models is not expected to help automatically. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
vote_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
vote_estimators_rows = []
for value in tqdm(vote_estimators_values, desc="VotingClassifier.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(vote_reference).set_params(estimators=fold_owned_members(preparation, value, memory=member_memory), weights=None)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    vote_estimators_rows.append({"configuration": f"VotingClassifier(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], voting={candidate.voting!r}" + ("" if candidate.weights is None else f", weights={candidate.weights}") + ")", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
vote_estimators_sweep = pd.DataFrame(vote_estimators_rows)
pd.concat([vote_estimators_sweep.nlargest(5, "cv_F1"), vote_estimators_sweep.nsmallest(5, "cv_F1")]).set_index("setting")[["members", "kinds", "train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
vote_estimators_winner = vote_estimators_sweep.loc[vote_estimators_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_estimators_winner["configuration"], "train": vote_estimators_winner["train"], "validation": vote_estimators_winner["validation"], "model": vote_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The best member subset in this three-fold sweep scores validation F1 0.8695: logistic regression + KNN + bagged trees + random forest + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 13:__ Count how many kinds of model each voting combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation F1 of the combinations by that number.
vote_by_kinds = vote_estimators_sweep.groupby("kinds")["cv_F1"].agg(["count", "mean", "min", "max"]).round(4)
display(vote_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", vote_estimators_sweep.nlargest(5, "cv_F1")["kinds"].tolist())

**Combinations that mix several kinds of model do not score higher in voting: the mean validation F1 is 0.8674 for the four combinations of one kind and between 0.8586 and 0.8595 for combinations of two to five kinds.** The kinds mixed by the best five are 3, 3, 1, 3 and 2.

__Step 14:__ Plot the validation F1 of every voting combination against its size and the kinds it mixes.

In [ ]:
# Validation F1 of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=vote_estimators_sweep.astype({"kinds": str}), x="members", y="cv_F1", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=vote_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation F1", title="Voting: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 15:__ Change only voting `weights` on the fixed training folds, using equal weights and each vector that doubles one member.

### `voting: weights`

Weights multiply each member's contribution; the default `None` gives equal weight. Emphasising a member is expected to pull the answer towards it. That helps only if its predictions deserve the extra influence. The candidates are equal weights and each emphasis vector that doubles one member while leaving the others at one.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
vote_weights_values = weight_values
# One row per value will hold its scores, its seconds and its fitted model.
vote_weights_rows = []
for value in tqdm(vote_weights_values, desc="VotingClassifier.weights", unit="value"):
    started = time.perf_counter()
    candidate = clone(vote_reference).set_params(weights=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    vote_weights_rows.append({"configuration": f"VotingClassifier(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], voting={candidate.voting!r}" + ("" if candidate.weights is None else f", weights={candidate.weights}") + ")", "setting": "equal" if value is None else "double " + members[value.index(2)][0], "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
vote_weights_sweep = pd.DataFrame(vote_weights_rows)
vote_weights_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
vote_weights_winner = vote_weights_sweep.loc[vote_weights_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_weights_winner["configuration"], "train": vote_weights_winner["train"], "validation": vote_weights_winner["validation"], "model": vote_weights_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**With the full member pool fixed, the best tested weighting scores validation F1 0.8633.** The weight table compares complete weighting vectors on the same rows.

__Step 16:__ Change only the vote's `voting` rule on the fixed training folds, with the sweep list `['hard', 'soft']`.

### `voting: voting`

Hard voting counts predicted labels; soft voting averages predicted probabilities. The library default is `hard`, while this explicit reference uses `soft`. Soft voting is expected to help when confidence estimates are informative; hard voting needs no probabilities. The candidates are `['hard', 'soft']`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
vote_voting_values = ['hard', 'soft']
# One row per value will hold its scores, its seconds and its fitted model.
vote_voting_rows = []
for value in tqdm(vote_voting_values, desc="VotingClassifier.voting", unit="value"):
    started = time.perf_counter()
    candidate = clone(vote_reference).set_params(voting=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    vote_voting_rows.append({"configuration": f"VotingClassifier(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], voting={candidate.voting!r}" + ("" if candidate.weights is None else f", weights={candidate.weights}") + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
vote_voting_sweep = pd.DataFrame(vote_voting_rows)
vote_voting_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
vote_voting_winner = vote_voting_sweep.loc[vote_voting_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_voting_winner["configuration"], "train": vote_voting_winner["train"], "validation": vote_voting_winner["validation"], "model": vote_voting_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Hard voting scores 0.8625 validation F1 and soft voting 0.8597.**

__Step 17:__ Select the best tested voting configuration from the existing sweep rows by validation score.

In [ ]:
vote_complete = pd.DataFrame(vote_estimators_rows + vote_weights_rows + vote_voting_rows)
vote_best = vote_complete.loc[vote_complete["cv_F1"].idxmax()]
print("best tested voting configuration:", vote_best["configuration"])

**The best complete vote measured combines bagged trees, the forest and the boosting winner by soft voting, with validation F1 0.8695.**

# <font color='#E8800A'>Stacking</font> <a class="anchor" id="stack"></a>
[Back to TOC](#toc)

A final learner sees member predictions as features. Its training predictions must come from members and preparation that did not fit the predicted row, so stacking cross-fits every training row. Stacking may draw on every algorithm taught so far.

__Step 18:__ Fit a toy final learner to out-of-fold prediction pairs and read its fitted combination.

In [ ]:
toy_meta = np.array([[0.1, 0.2], [0.4, 0.7], [0.8, 0.9], [0.7, 0.3]])
toy_meta_response = np.array([0, 1, 1, 0])
toy_final = LogisticRegression().fit(toy_meta, toy_meta_response)
print("toy member predictions:", toy_meta)
print("toy final positive probabilities:", toy_final.predict_proba(toy_meta)[:, 1].round(3))

**The toy final learner turns the four prediction pairs into positive probabilities 0.443, 0.518, 0.559 and 0.48.**

These small arrays stand for predictions made without fitting their own rows; they are not predictions copied from fitted training responses. A same-row construction is forbidden and is not executed. On the course data the helper splits raw rows first, cross-fits preparation and members, trains the final learner on the resulting predictions, and only then refits members for deployment.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html'>sklearn.ensemble.StackingClassifier(estimators, final_estimator=None, cv=None, stack_method='auto', passthrough=False, n_jobs=None)</a>

**Definition:**
Fits a final learner on the out-of-fold predictions of its members, so that the final learner learns how to combine them.

**Main Parameters:**
- `estimators`: the named members.
- `final_estimator`: the learner fitted on the member predictions.
- `cv`: the internal folds that produce the out-of-fold predictions.
- `stack_method`: the member output used as a feature.
- `passthrough`: whether the final learner also sees the original columns.

**Main Attributes (after fitting):**
- `estimators_`: the members refitted on all rows.
- `final_estimator_`: the fitted final learner.

**Main Methods:**
- `fit(X, y)`: cross-fit the members, fit the final learner and refit the members.
- `predict_proba(X)`: the final learner's probabilities.
</div>

[Wolpert (1992)](https://doi.org/10.1016/S0893-6080%2805%2980023-1) introduced stacked generalization. [`StackingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html) and the [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) define the method. `FoldStacking`, in `preprocessing.py`, matches its out-of-fold predictions and final learner, as the check below shows. Wrapping each scikit-learn member with preparation gives the same fold-owned preparation. The course helper caches each member's predictions once per internal fold for reuse across the 120 combinations and the combiner search; scikit-learn would refit members per combination. Its shared interface also supports blending, which scikit-learn does not implement.

`passthrough=False` is fixed here: the final learner sees only positive-class probabilities. Base preparation is fitted inside each internal fold. If passthrough is enabled, a separate preprocessor fitted on this stacking fit's training rows provides one stable feature schema, appending prepared features rather than scikit-learn's raw inputs. The equality check uses `passthrough=False`. Boosting final learners receive member predictions without preparation.

<div class="alert alert-block alert-info">
<code>FoldStacking(preprocessor, estimators, final_estimator, *, cv=3, method="auto", passthrough=False, random_state=33, memory=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
Cross-fits raw-row members: each internal fit predicts its score rows. The final learner fits the collected out-of-fold predictions, then members are refitted on all rows.

**Main Parameters:**
- `preprocessor`: unfitted recipe, fitted on each member fit's training rows.
- `estimators`: `(name, learner)` pairs wrapped by `fold_owned_members`.
- `final_estimator`: learner fitted on member predictions.
- `cv`: internal fold count; shuffled `StratifiedKFold` for classifiers, `KFold` otherwise.
- `method`: `"auto"` uses `predict_proba`'s second column, the positive-class probability.
- `passthrough`: `False` uses only member predictions. `True` appends features from a separate `preprocessor` clone fitted on this stacking fit's training rows and reused at prediction.
- `random_state`: internal split seed.
- `memory`: optional joblib `Memory` caching member predictions.

**Main Attributes (after fit):**
- `method_`: response method used.
- `ownership_`: fit and score row labels for every internal fold.
- `meta_estimator_`: fitted final learner.
- `estimators_`: members refitted on all rows.
- `passthrough_preprocessor_`: fitted feature preparation when passthrough is enabled.

**Main Methods:**
- `fit(X, y)`: cross-fit members, fit the final learner and refit members.
- `predict(X)`: final learner's prediction.
- `predict_proba(X)`: final learner's probabilities, for classifiers.

**Differences from scikit-learn:**
The check below compares [`StackingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html), described in the [user guide](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization), on the same rows with `passthrough=False`: both produce the same out-of-fold predictions and final learner. `FoldStacking` adds preparation inside internal fits and prediction caching. Its passed-through features use one training-owned schema; scikit-learn passes original inputs. It uses positive-class probabilities, whereas `StackingClassifier` selects outputs through `stack_method`.
</div>

__Step 19:__ Fit scikit-learn `StackingClassifier` and `FoldStacking` on the same rows and compare their predictions.

In [ ]:
# A small member set and the first fold of raw rows, so the two stackings can be compared quickly.
equal_members = [(name, taught_learners[name]) for name in ["logistic regression", "KNN", "pruned tree"]]
equal_fold = raw_prepared[0]
equal_splitter = StratifiedKFold(3, shuffle=True, random_state=random_state)
# scikit-learn's stacking: every member is wrapped with the recipe so it prepares its own rows, and the same three-fold splitter produces the out-of-fold predictions.
started = time.perf_counter()
library_stack = StackingClassifier([(name, PreparedEstimator(FoldPreprocessor(recipe), clone(member))) for name, member in equal_members],
    final_estimator=clone(taught_learners["logistic regression"]), cv=equal_splitter, stack_method="predict_proba", passthrough=False, n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
library_seconds = time.perf_counter() - started
# The course stacking on the same rows, members and splits, with no cache.
started = time.perf_counter()
course_stack = FoldStacking(preparation, equal_members, clone(taught_learners["logistic regression"]), cv=3, random_state=random_state).fit(equal_fold["X_fit"], equal_fold["y_fit"])
course_seconds = time.perf_counter() - started
# The sweeps reuse member predictions through a cache: one fit fills it and the next, timed, fit reads it.
equal_memory = joblib.Memory(".equal_stack_cache", verbose=0)
FoldStacking(preparation, equal_members, clone(taught_learners["logistic regression"]), cv=3, random_state=random_state, memory=equal_memory).fit(equal_fold["X_fit"], equal_fold["y_fit"])
started = time.perf_counter()
cached_stack = FoldStacking(preparation, equal_members, clone(taught_learners["logistic regression"]), cv=3, random_state=random_state, memory=equal_memory).fit(equal_fold["X_fit"], equal_fold["y_fit"])
cached_seconds = time.perf_counter() - started
# Recompute the out-of-fold predictions with `cross_val_predict` under the same splitter and fit the final learner on them: its coefficients must match both stackings.
out_of_fold = np.column_stack([cross_val_predict(PreparedEstimator(FoldPreprocessor(recipe), clone(member)), equal_fold["X_fit"], equal_fold["y_fit"],
    cv=equal_splitter, method="predict_proba")[:, 1] for _, member in tqdm(equal_members, desc='equal members', leave=True, position=0)])
reference_final = clone(taught_learners["logistic regression"]).fit(out_of_fold, equal_fold["y_fit"])
same_out_of_fold = np.array_equal(reference_final.coef_, library_stack.final_estimator_.coef_) and np.array_equal(reference_final.coef_, course_stack.meta_estimator_.coef_)
# Held-out rows go through the refitted members and the final learner of each stacking; they must give the same answers.
library_held_out = library_stack.predict_proba(equal_fold["X_score"])
course_held_out = course_stack.predict_proba(equal_fold["X_score"])
same_held_out = np.array_equal(library_held_out, course_held_out)
# Stop here if the two stackings differ, and print what matched with the three timings.
assert same_out_of_fold and same_held_out, (same_out_of_fold, np.abs(library_held_out - course_held_out).max())
print("out-of-fold predictions give the same final learner in both stacks:", same_out_of_fold)
print("held-out predictions are identical:", same_held_out, "| largest difference", np.abs(library_held_out - course_held_out).max())
print(f"StackingClassifier {library_seconds:.2f} s, FoldStacking {course_seconds:.2f} s, FoldStacking with a warm member cache {cached_seconds:.2f} s")

**The two implementations reproduce the same held-out predictions within the stated tolerance.** The cached version reuses member predictions for the same training and scoring rows; it does not alter the stacking protocol.

<div class="alert alert-block alert-info">
<code>unprepared_boosting(estimator)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:**
Returns an unfitted boosted learner for prediction columns. If its base is a `SampleWeightedPreparedEstimator`, the copy uses the enclosed plain learner, wrapped in `CalibratedClassifierCV` with the same method when calibrated. Otherwise it returns `clone(estimator)`.

**Main Parameters:**
- `estimator`: boosted learner to copy.

**Returns:**
An unfitted estimator.

**Differences from scikit-learn:**
`clone` retains a base learner's raw-row preparation. This helper removes it because a stack's final learner receives member predictions as columns.
</div>

__Step 20:__ Declare the member combinations and the final learners before any combination score is seen.

The members are the 120 combinations of two to seven models of the pool that the vote uses, and the final learners are the seven models of the pool; no combination of single-setting winners and no test score enters. The combinations are compared side by side on the first three training folds with logistic regression as the final learner. Stacking fixes its internal fold count at 3, because each member is refitted once per internal fold.

In [ ]:
final_learner_values = list(taught_learners.values()) + [unprepared_boosting(boosting_member)]
stack_reference = FoldStacking(preparation, members, clone(taught_learners["logistic regression"]), cv=3, random_state=random_state, memory=member_memory)

__Step 21:__ Score the FoldStacking reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(stack_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
stack_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
stack_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 22:__ Fit the FoldStacking reference on all the training rows.

In [ ]:
stack_default = clone(stack_reference).fit(X_train, y_train)
stack_default_configuration = f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in stack_reference.estimators)}], final_estimator={' '.join(str(stack_reference.final_estimator).split())})"

__Step 23:__ Log the FoldStacking reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_default_configuration,
    "train": stack_default_train, "validation": stack_default_validation, "model": stack_default},
    X_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Stacking settings</font> <a class="anchor" id="stack_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the stack and scores every value on the first three training folds, because every combination refits all of its members on each internal fold. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members whose out-of-fold predictions feed the final learner. |
| `final_estimator` | Learner fitted on those predictions. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)

__Step 24:__ Change only stacking `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `stacking: estimators`

Out-of-fold member predictions become columns for the final learner. There is no default member list: the full pool is fixed before the sweep, as are the final learner and three internal folds. Extra members are expected to help only when their predictions add useful information. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
stack_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
stack_estimators_rows = []
for value in tqdm(stack_estimators_values, desc="FoldStacking.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(stack_reference).set_params(estimators=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    stack_estimators_rows.append({"configuration": f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())})", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
stack_estimators_sweep = pd.DataFrame(stack_estimators_rows)
pd.concat([stack_estimators_sweep.nlargest(5, "cv_F1"), stack_estimators_sweep.nsmallest(5, "cv_F1")]).set_index("setting")[["members", "kinds", "train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
stack_estimators_winner = stack_estimators_sweep.loc[stack_estimators_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_estimators_winner["configuration"], "train": stack_estimators_winner["train"], "validation": stack_estimators_winner["validation"], "model": stack_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The best member subset in this three-fold sweep scores validation F1 0.8552: naive Bayes + KNN + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 25:__ Count how many kinds of model each stacking combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation F1 of the combinations by that number.
stack_by_kinds = stack_estimators_sweep.groupby("kinds")["cv_F1"].agg(["count", "mean", "min", "max"]).round(4)
display(stack_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", stack_estimators_sweep.nlargest(5, "cv_F1")["kinds"].tolist())

**Stacks that mix more kinds of model score higher: the mean validation F1 rises from 0.8026 for one kind to 0.8119 for two, 0.8329 for three, 0.8441 for four and 0.8484 for five.** The best five stacks mix three or four kinds, three in the best three. A combination of k kinds has at least k members, so size and kinds rise together in this table.

__Step 26:__ Plot the validation F1 of every stacking combination against its size and the kinds it mixes.

In [ ]:
# Validation F1 of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=stack_estimators_sweep.astype({"kinds": str}), x="members", y="cv_F1", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=stack_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation F1", title="Stacking: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 27:__ Score each stack member by itself on the same three folds, beside the four stacks.

In [ ]:
# Each stack member scored alone on the same three folds. The members reuse the predictions the stack already cached, so nothing is refitted.
stack_member_rows = []
for name, member in tqdm(fold_owned_members(preparation, members, memory=member_memory), desc='stack members', unit='member', leave=True, position=0):
    started = time.perf_counter()
    # Scores on the rows each member was fitted on and on the rows it did not see.
    train_scores, validation_scores = [], []
    for fold in combiner_folds:
        fitted = clone(member).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], (fitted.predict_proba(fold["X_fit"])[:, 1] > 0.5).astype(int)))
        validation_scores.append(f1_score(fold["y_score"], (fitted.predict_proba(fold["X_score"])[:, 1] > 0.5).astype(int)))
    # The member fitted on all the training rows is scored once on the sealed test rows.
    fitted = clone(member).fit(X_train, y_train)
    stack_member_rows.append({"setting": name, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, (fitted.predict_proba(X_test)[:, 1] > 0.5).astype(int)), "seconds": time.perf_counter() - started})
# Best member first, so the table reads against the stack rows above.
stack_member_scores = pd.DataFrame(stack_member_rows).sort_values("cv_F1", ascending=False)
stack_member_scores.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

**The best member alone, bagged trees, scores validation F1 0.8653, above the best stack in the table above, 0.8552.** The forest alone scores 0.8650 and the boosting winner 0.8628, so no stack here beats its best member.

__Step 28:__ Change only stacking `final_estimator` on the fixed training folds, using the seven models of the pool.

### `stacking: final_estimator`

The final learner fits out-of-fold member predictions. The default `None` selects scikit-learn's `LogisticRegression`; this course reference explicitly fixes the taught logistic regression. A flexible final learner is expected to capture interactions between member predictions but may overfit these few input columns. The candidates are the six earlier models and the boosting winner.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
stack_final_estimator_values = final_learner_values
# One row per value will hold its scores, its seconds and its fitted model.
stack_final_estimator_rows = []
for value in tqdm(stack_final_estimator_values, desc="FoldStacking.final_estimator", unit="value"):
    started = time.perf_counter()
    candidate = clone(stack_reference).set_params(final_estimator=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    stack_final_estimator_rows.append({"configuration": f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())})", "setting": (list(taught_learners) + ["boosting winner"])[final_learner_values.index(value)], "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
stack_final_estimator_sweep = pd.DataFrame(stack_final_estimator_rows)
stack_final_estimator_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
stack_final_estimator_winner = stack_final_estimator_sweep.loc[stack_final_estimator_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_final_estimator_winner["configuration"], "train": stack_final_estimator_winner["train"], "validation": stack_final_estimator_winner["validation"], "model": stack_final_estimator_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**With all members fixed, the best final learner is KNN, scoring validation F1 0.8661.** This comparison does not inherit the member subset selected in the preceding sweep.

__Step 29:__ Select the best tested stacking configuration from the existing sweep rows by validation score.

In [ ]:
stack_complete = pd.DataFrame(stack_estimators_rows + stack_final_estimator_rows)
stack_best = stack_complete.loc[stack_complete["cv_F1"].idxmax()]
print("best tested stacking configuration:", stack_best["configuration"])

**The best complete stack configuration actually evaluated scores validation F1 0.8661.** Its full settings are shown in the table; separate sweep winners are not combined without evaluation.

# <font color='#E8800A'>Blending</font> <a class="anchor" id="blend"></a>
[Back to TOC](#toc)

Blending reserves one slice of the training rows for the final learner. The members never see that slice, even after deployment, so its predictions are always made on unseen rows.

__Step 30:__ Fit a toy final learner to the predictions made on a held-out slice.

In [ ]:
toy_hold = np.array([[0.2, 0.1], [0.9, 0.7], [0.4, 0.8], [0.8, 0.9]])
toy_hold_response = np.array([0, 1, 0, 1])
toy_blender = LogisticRegression().fit(toy_hold, toy_hold_response)
print("held-out member predictions:", toy_hold)
print("blended positive probabilities:", toy_blender.predict_proba(toy_hold)[:, 1].round(3))

**The toy final learner turns the four held-out prediction pairs into positive probabilities 0.416, 0.545, 0.491 and 0.547.**

The slice is cut from raw rows before any member or preparation is fitted. Blending uses the same 120 combinations as stacking.

<div class="alert alert-block alert-info">
<code>FoldBlending(preprocessor, estimators, final_estimator, *, fraction=0.25, method="auto", random_state=33, memory=None)</code>

A course subclass of `FoldStacking` in `preprocessing.py`, beside this notebook.

**Definition:**
Holds out raw training rows for the final learner. Members fit the remaining rows and predict the held-out rows, then the final learner fits those predictions. Members are never refitted on the held-out rows.

**Main Parameters:**
- `preprocessor`: unfitted recipe, fitted inside every member fit.
- `estimators`: `(name, learner)` pairs wrapped by `fold_owned_members`.
- `final_estimator`: learner fitted on member predictions.
- `fraction`: held-out training share; classifiers use a stratified split.
- `method`: `"auto"` uses `predict_proba`'s second column, the positive-class probability.
- `random_state`: split seed.
- `memory`: optional joblib `Memory` caching member predictions.

**Main Attributes (after fit):**
- `method_`: response method used.
- `ownership_`: member-fit and held-out row labels.
- `meta_estimator_`: fitted final learner.
- `estimators_`: members fitted on the remaining rows.

**Main Methods:**
- `fit(X, y)`: fit members, then the final learner on held-out predictions.
- `predict(X)`: final learner's prediction.
- `predict_proba(X)`: final learner's probabilities, for classifiers.

**Differences from scikit-learn:**
This course holdout variant has no scikit-learn estimator. The [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) describes the related stacking method. Stacking trains its final learner on every row's out-of-fold predictions; blending uses one held-out slice. Sharing `FoldStacking`'s interface keeps the constructions comparable.
</div>

__Step 31:__ Construct the blend reference, which holds a quarter of the training rows out for the final learner.

In [ ]:
blend_reference = FoldBlending(preparation, members, clone(taught_learners["logistic regression"]), fraction=0.25, random_state=random_state, memory=member_memory)

__Step 32:__ Score the FoldBlending reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(blend_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
blend_default_train = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
blend_default_validation = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 33:__ Fit the FoldBlending reference on all the training rows.

In [ ]:
blend_default = clone(blend_reference).fit(X_train, y_train)
blend_default_configuration = f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in blend_reference.estimators)}], final_estimator={' '.join(str(blend_reference.final_estimator).split())}, fraction={blend_reference.fraction})"

__Step 34:__ Log the FoldBlending reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_default_configuration,
    "train": blend_default_train, "validation": blend_default_validation, "model": blend_default},
    X_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Blending settings</font> <a class="anchor" id="blend_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the blend and scores every value on the first three training folds, because every combination refits all of its members. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members fitted on the rows left after the held-out slice. |
| `fraction` | Share of training rows held out for the final learner. |

`FoldBlending` is the course holdout variant in `preprocessing.py`, beside `FoldStacking`. The [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) describes the related stacking method.

__Step 35:__ Change only blending `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `blending: estimators`

Members fit one training subset and predict a separate subset used by the final learner. There is no default member list: the full pool, final learner and holdout fraction form the fixed reference. A new member is expected to help only when it adds information the others lack. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
blend_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
blend_estimators_rows = []
for value in tqdm(blend_estimators_values, desc="FoldBlending.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(blend_reference).set_params(estimators=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    blend_estimators_rows.append({"configuration": f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())}, fraction={candidate.fraction})", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
blend_estimators_sweep = pd.DataFrame(blend_estimators_rows)
pd.concat([blend_estimators_sweep.nlargest(5, "cv_F1"), blend_estimators_sweep.nsmallest(5, "cv_F1")]).set_index("setting")[["members", "kinds", "train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
blend_estimators_winner = blend_estimators_sweep.loc[blend_estimators_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_estimators_winner["configuration"], "train": blend_estimators_winner["train"], "validation": blend_estimators_winner["validation"], "model": blend_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**The best member subset in this three-fold sweep scores validation F1 0.8615: naive Bayes + KNN + pruned tree + bagged trees + random forest + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 36:__ Count how many kinds of model each blending combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation F1 of the combinations by that number.
blend_by_kinds = blend_estimators_sweep.groupby("kinds")["cv_F1"].agg(["count", "mean", "min", "max"]).round(4)
display(blend_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", blend_estimators_sweep.nlargest(5, "cv_F1")["kinds"].tolist())

**Blends that mix more kinds of model score higher: the mean validation F1 rises from 0.8025 for one kind to 0.8131 for two, 0.8353 for three, 0.8471 for four and 0.8517 for five.** The kinds mixed by the best five are 4, 4, 4, 4 and 5.

__Step 37:__ Plot the validation F1 of every blending combination against its size and the kinds it mixes.

In [ ]:
# Validation F1 of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=blend_estimators_sweep.astype({"kinds": str}), x="members", y="cv_F1", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=blend_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation F1", title="Blending: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 38:__ Change only blending `fraction` on the fixed training folds, with the sweep list `[0.2, 0.25, 0.33]`.

### `blending: fraction`

This fraction of training rows is reserved for fitting the final learner. The course class defaults to `0.25` and has no scikit-learn equivalent. A larger fraction is expected to improve estimation of the combining rule while weakening members trained on fewer rows. The candidate list is `[0.2, 0.25, 0.33]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
blend_fraction_values = [0.2, 0.25, 0.33]
# One row per value will hold its scores, its seconds and its fitted model.
blend_fraction_rows = []
for value in tqdm(blend_fraction_values, desc="FoldBlending.fraction", unit="value"):
    started = time.perf_counter()
    candidate = clone(blend_reference).set_params(fraction=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [f1_score(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    blend_fraction_rows.append({"configuration": f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())}, fraction={candidate.fraction})", "setting": value, "seconds": time.perf_counter() - started,
        "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
blend_fraction_sweep = pd.DataFrame(blend_fraction_rows)
blend_fraction_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
blend_fraction_winner = blend_fraction_sweep.loc[blend_fraction_sweep["cv_F1"].idxmax()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_fraction_winner["configuration"], "train": blend_fraction_winner["train"], "validation": blend_fraction_winner["validation"], "model": blend_fraction_winner["model"]},
    X_test, y_test, "one setting changed", metric=f1_score, label="F1")

**With the full member pool fixed, fraction 0.2 gives the highest tested validation F1, 0.8628.** The other settings remain unchanged.

__Step 39:__ Select the best tested blending configuration from the existing sweep rows by validation score.

In [ ]:
blend_complete = pd.DataFrame(blend_estimators_rows + blend_fraction_rows)
blend_best = blend_complete.loc[blend_complete["cv_F1"].idxmax()]
print("best tested blending configuration:", blend_best["configuration"])

**The best complete blend configuration actually evaluated scores validation F1 0.8628.** Its full settings are shown in the table; separate sweep winners are not combined without evaluation.

__Step 40:__ Audit the stack and blend row ownership from the two default fits.

In [ ]:
for label_owned, construction in [("stack", stack_default), ("blend", blend_default)]:
    overlaps = [len(set(a) & set(b)) for a, b in construction.ownership_]
    print(label_owned, "internal fit/score overlap counts:", overlaps)
    assert not any(overlaps)
print("stack meta-training coverage:", sum(len(b) for a, b in stack_default.ownership_))
print("blend member rows:", len(blend_default.ownership_[0][0]), "meta rows:", len(blend_default.ownership_[0][1]))

# <font color='#E8800A'>Selection of the combiners</font> <a class="anchor" id="sel2"></a>
[Back to TOC](#toc)

The combiner search tests voting, stacking and blending over the pool subsets and swept settings. Nested cross-validation evaluates this selection first. In each of its ten outer folds, the boosting member uses settings selected on that fold's training rows. The final search uses all training rows and the final boosting winner.

The grid covers voting, stacking and blending: every combination of two to seven members and every swept setting, with the weights only where the members are the full seven. The stored winning configurations of the earlier models are fixed teaching inputs, not a fresh outer-fold search over their original tuning histories. As before, the search evaluates 60 candidates, drawn uniformly without replacement from the enumerated space with a seed of 33 before any score is seen, and the same 60 feed the nested estimate and the search on all the training rows.

__Step 41:__ Declare the conditional grids of the combiners, which fix the search space before any search score exists.

In [ ]:
# The search space of the second selection: voting, stacking and blending over the 120 combinations of two to seven pool models and their swept settings, with the weights only on the full pool of seven.
# A member is a position in the pool, so that each outer fold can put its own boosting winner in the last place.
combiner_grid = [
    # Voting: every member combination, and separately the weights, which only fit the full pool; its members prepare their own rows.
    {"preprocessor": [FunctionTransformer()], "estimator": [clone(vote_reference)],
     "estimator__estimators": member_index,
     "estimator__voting": vote_voting_values},
    {"preprocessor": [FunctionTransformer()], "estimator": [clone(vote_reference)],
     "estimator__estimators": [tuple(range(len(members)))],
     "estimator__weights": vote_weights_values,
     "estimator__voting": vote_voting_values},
    # Stacking: every member combination with every final learner, given as a position in the pool.
    {"preprocessor": [FunctionTransformer(kw_args={})], "estimator": [clone(stack_reference)],
     "estimator__estimators": member_index,
     "estimator__final_estimator": list(range(len(final_learner_values)))},
    # Blending: every member combination with every held-out fraction.
    {"preprocessor": [FunctionTransformer(inv_kw_args={})], "estimator": [clone(blend_reference)],
     "estimator__estimators": member_index,
     "estimator__fraction": blend_fraction_values}
]

__Step 42:__ Enumerate every valid combination of the combiners, and draw 60 of them without replacement before any score is seen.

In [ ]:
# The conditional grids hold every valid combination; the search draws 60 of them, uniformly and without replacement, before any score exists.
print("combinations by group:", [len(ParameterGrid(group)) for group in combiner_grid])
print("combinations in the space:", len(ParameterGrid(combiner_grid)))

**The conditional grids of the combiners hold 1456 valid combinations, and the search evaluates 60 of them.**

__Step 43:__ Give each outer fold its own boosting member, and turn the drawn combinations into the candidates of that fold.

In [ ]:
# One boosting member for each outer fold, from the settings that fold's search chose in the first selection, and one for the search on all the training rows.
boost_members = [clone(chosen["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in chosen.items() if key.startswith("estimator__")})
    for chosen in boost_fold_settings + [boost_settings]]
# For each of them the grid keeps its shape: a position in the pool becomes the model at that position, with that fold's boosting member last.
combiner_grids = []
for boost_member in boost_members:
    pool = list(taught_learners.items()) + [("boosting winner", boost_member)]
    finals = list(taught_learners.values()) + [unprepared_boosting(boost_member)]
    fold_grid = []
    for group in combiner_grid:
        # A vote takes members that prepare their own rows; stacking and blending prepare inside their own splits and take the plain models.
        voting = type(group["estimator"][0]).__name__.startswith("Voting")
        fold_grid.append({name: ([fold_owned_members(preparation, [pool[i] for i in value], memory=member_memory) if voting else [pool[i] for i in value] for value in values]
                                 if name == "estimator__estimators" else [finals[value] for value in values] if name == "estimator__final_estimator" else values)
                          for name, values in group.items()})
    combiner_grids.append(fold_grid)

__Step 44:__ Wrap each grid and the preprocessing in one search, with the same three inner folds.

In [ ]:
combiner_searches = [CachedSearchCV(RandomizedSearchCV(PreparedEstimator(FunctionTransformer(), clone(vote_reference)), grid, n_iter=60, random_state=random_state, cv=inner, scoring="f1", n_jobs=4)) for grid in combiner_grids]

__Step 45:__ Estimate the whole search with the ten outer folds of the training partition, returning scores rather than models.

In [ ]:
combiner_nested_started = time.perf_counter()
# Each of the ten outer folds runs its own search, with that fold's boosting winner as the boosting member, one fold per worker; the search inside uses four workers.
outer_runs = joblib.Parallel(n_jobs=10, return_as="generator")(
    joblib.delayed(cross_validate)(combiner_searches[position], X_train, y_train, cv=[split],
        scoring=companions, return_train_score=True, n_jobs=1, error_score="raise")
    for position, split in enumerate(folds.split(X_train, y_train)))
# Collect the outer results as the workers finish; the progress bar fills in bursts.
combiner_fold_results = list(tqdm(outer_runs, total=folds.get_n_splits(),
                                  desc="Nested CV, combiners: outer folds", unit="fold", leave=True, position=0))
# The outer validation scores estimate the whole procedure, selection step included; the training scores sit beside them, and ROC-AUC and balanced accuracy come from the same outer fits.
combiner_nested_scores = {key: np.concatenate([result[key] for result in combiner_fold_results]) for key in ("train_f1", "test_f1", "test_roc_auc", "test_balanced_accuracy")}
combiner_nested_validation = combiner_nested_scores["test_f1"]
combiner_nested_train = combiner_nested_scores["train_f1"]
combiner_nested_seconds = time.perf_counter() - combiner_nested_started
# Report the estimate, its spread over the folds and the time.
print(f"nested validation F1: {combiner_nested_validation.mean():.4f}")
print(f"fold SD: {combiner_nested_validation.std(ddof=1):.4f}")
# ROC-AUC averages over the outer folds whose winner gives probabilities; balanced accuracy covers all of them.
print(f"ROC-AUC over the {np.isfinite(combiner_nested_scores['test_roc_auc']).sum()} of {len(combiner_nested_scores['test_roc_auc'])} outer folds whose winner gives probabilities")
print(f"nested wall time: {combiner_nested_seconds:.2f} seconds")

**The complete search scores a nested validation F1 of 0.8670 with a fold SD of 0.0137.** The space holds 1456 valid combinations, 60 of them drawn for the search. ROC-AUC averages over the 5 of 10 outer folds whose winner gives probabilities, because a hard vote gives none; balanced accuracy covers all 10 folds.

# <font color='#E8800A'>The combiner winner</font> <a class="anchor" id="combiner_winner"></a>
[Back to TOC](#toc)

With the search space fixed and its nested estimate available, the same candidates are fitted to all the training rows, with the boosting winner of the first selection as the boosting member. Its best estimator is the combiner winner.

__Step 46:__ Run the declared grid of the combiners on all the training rows and keep its winner.

In [ ]:
combiner_started = time.perf_counter()
combiner_search = combiner_searches[-1].fit(X_train, y_train)
combiner_winner = combiner_search.best_estimator_
combiner_seconds = time.perf_counter() - combiner_started
print("chosen parameters:", combiner_search.best_params_)
print("winner own inner-CV score:", combiner_search.best_score_)
print("combinations evaluated:", len(combiner_search.cv_results_["params"]))
print(f"search wall time: {combiner_seconds:.2f} seconds")

__Step 47:__ Score the combiner winner on the ten outer folds, and log it beside its test score.

In [ ]:
# Score the winner on the same folds: stacking and blending split their own rows, so each fold's raw training rows go to the combiner.
combiner_train, combiner_validation = [], []
for fold in tqdm(raw_prepared, desc='raw folds', leave=True, position=0):
    fitted = clone(combiner_winner.estimator).fit(fold["X_fit"], fold["y_fit"])
    combiner_train.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
    combiner_validation.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))

In [ ]:
combiner_settings = combiner_search.best_params_
combiner_constructor = clone(combiner_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in combiner_settings.items() if key.startswith("estimator__")})
# Name the winner by its members and the settings that differ from the defaults.
combiner_label = (f"{type(combiner_constructor).__name__}(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in combiner_constructor.estimators)}], "
    + ", ".join(f"{key.split('estimator__', 1)[1]}={' '.join(str(value).split())}" for key, value in combiner_settings.items()
                if key.startswith("estimator__") and key != "estimator__estimators") + ")")
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": combiner_label, "train": combiner_train, "validation": combiner_validation, "model": combiner_winner},
    X_test, y_test, "winner's own score", metric=f1_score, label="F1")

__Step 48:__ Log the combiner winner with its nested estimate as the validation score.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": f"combiner search over {len(combiner_search.cv_results_['params'])} combinations", "train": combiner_nested_train, "validation": combiner_nested_validation, "model": combiner_winner},
    X_test, y_test, "nested estimate", metric=f1_score, label="F1")

__Step 49:__ Read the test score of the combiner winner from the log.

In [ ]:
combiner_test = model_log[-1]["test F1"]

**The search chose a stack of the pruned tree and the forest with a pruned decision tree as its final learner, with held-out F1 0.8762 and an inner-CV score of 0.8695.** Its own score over the ten outer folds is 0.8660, against a nested estimate of 0.8670 for the whole search.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The nested estimate describes the declared selection procedure. The test score belongs to the actual final-grid winner. Earlier results join the board only when their data and protocol fingerprints match.

__Step 50:__ Build the Dummy floor on training rows, then join the saved earlier results.

In [ ]:
board = Scoreboard("champions Outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train)),
    floor=f1_score(y_test, DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)))
for week, earlier_model, earlier_cv, earlier_test in results_before(11, "classification", results_dir, metric=board.metric, context=board.context):
    board.record(week, earlier_model, earlier_cv, earlier_test)

__Step 51:__ Save the nested estimate, the winner test score and its complete constructor settings.

In [ ]:
# The week's row is the selection with the better nested estimate; the winner of the other selection stays in the log.
selections = {"boosting": {"nested": boost_nested_validation, "test": boost_test, "label": boost_label, "model": boosting_winner, "scores": boost_nested_scores},
    "combiner": {"nested": combiner_nested_validation, "test": combiner_test, "label": combiner_label, "model": combiner_winner, "scores": combiner_nested_scores}}
board_selection = "combiner" if combiner_nested_validation.mean() > boost_nested_validation.mean() else "boosting"
board_nested = selections[board_selection]["nested"]
board_test = selections[board_selection]["test"]
chosen_label = selections[board_selection]["label"]
chosen_model = selections[board_selection]["model"]
board_scores = selections[board_selection]["scores"]
print("Boosting nested F1: %.4f" % np.mean(boost_nested_validation))
print("Combining nested F1: %.4f" % np.mean(combiner_nested_validation))
print("board selection:", board_selection)
# Fixed settings and their own family reference use the same course folds.
if board_selection == "boosting":
    tuning_comparison = selection["tuning_comparison"]
else:
    selected_family = type(chosen_model.estimator).__name__
    reference_row = next(row for row in model_log
        if row.get("validation F1 is") == "untuned" and row["model"].startswith(selected_family + "("))
    candidate_row = next(row for row in model_log
        if row.get("validation F1 is") == "winner's own score" and row["model"] == chosen_label)
    tuning_comparison = dict(
        protocol="fixed_settings_on_course_folds", family=selected_family,
        candidate_model=chosen_label, reference_model=reference_row["model"],
        reference_kind="declared_reference", candidate_cv=float(candidate_row["validation F1"]),
        reference_cv=float(reference_row["validation F1"]),
        candidate_test=float(candidate_row["test F1"]), reference_test=float(reference_row["test F1"]))
comparison_reference_cv = tuning_comparison["reference_cv"]
comparison_reference_test = tuning_comparison["reference_test"]
# Record the nested estimate as the cross-validated number and the held-out score as the test number; save the row with its baseline and the fitted winner.
board.record(11, chosen_label, round(board_nested.mean(), 4), round(board_test, 4))
# F1 selects and publishes the row; ROC-AUC and balanced accuracy are companions measured on the same fitted models, with no extra fits.
winner_probability = chosen_model.predict_proba(X_test)[:, 1]
companion_metrics = pd.DataFrame({
    "nested outer-fold mean": {"F1": board_nested.mean(), "ROC-AUC": np.nanmean(board_scores["test_roc_auc"]),
                               "balanced accuracy": board_scores["test_balanced_accuracy"].mean()},
    "held-out winner": {"F1": board_test, "ROC-AUC": roc_auc_score(y_test, winner_probability),
                        "balanced accuracy": balanced_accuracy_score(y_test, chosen_model.predict(X_test))}}).round(4)
display(companion_metrics)
board.save(results_dir, 11, "classification", baseline_cv=round(comparison_reference_cv, 4), baseline_test=round(comparison_reference_test, 4))
joblib.dump(chosen_model, f"{results_dir}/week_11_classification_model.joblib")
# Print the board with every earlier result and the no-model floor.
print(board.table())
# Preserve both the nested procedure estimate and the fixed-setting comparison.
result_path = Path(results_dir) / "week_11.json"
result_document = json.loads(result_path.read_text(encoding="utf-8"))
result_document["classification"]["tuning_comparison"] = tuning_comparison
result_path.write_text(json.dumps(result_document, indent=1) + "\n", encoding="utf-8")


**The week's row takes the combiner selection, whose nested estimate of 0.8670 is higher than the 0.8620 of the boosting selection.** The row scores 0.8670 cross-validated and 0.8762 held out, against 0.8661 and 0.8816 for the forest row above: the combiner row is higher cross-validated and lower held out, and the boosting row is lower on both. ROC-AUC and balanced accuracy are companions of F1 on the same models: the nested ROC-AUC is 0.8977, averaged over the 5 of 10 outer folds whose winner gives probabilities, against 0.8772 held out, and balanced accuracy is 0.8101 nested over all 10 folds and 0.8144 held out. The no-model floor is 0.7529.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. Voting combines predictions with a fixed rule; stacking learns its rule from out-of-fold predictions.
2. Blending reserves training rows for the final learner, reducing the data available to its members.
3. Member lists and final learners are explicit fixed references: each isolated sweep changes one setting.
4. Each outer combiner search uses the boosting settings selected on that outer training fold.
5. The overall Week 11 row compares the two nested selection procedures.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

The boosting search has nested validation F1 0.8620 and the combining search 0.8670. Why must each outer combining fold use its own boosting selection?

**Answer.** The boosting member must be selected using only that outer fold's training rows. Loading the winner selected on all training rows would let the outer validation rows influence the member selection.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

Original papers support the method explanations; library documentation defines the estimator contracts.

- [scikit-learn ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)
- [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)
- [Wolpert, D. H. (1992). Stacked Generalization. Neural Networks, 5(2), 241-259.](https://doi.org/10.1016/S0893-6080%2805%2980023-1)
- [Bergstra, J. and Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. Journal of Machine Learning Research, 13(10), 281-305.](https://jmlr.org/papers/v13/bergstra12a.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [joblib disk-based caching](https://joblib.readthedocs.io/en/latest/user_guide/memory.html)
- [joblib serialization and loading](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html)